# INTRO

This notebook introduces the basic least-squares adjustment study based on GPS SATELLITE SURVEYING Fourth Edition by ALFRED LEICK, LEV RAPOPORT, DMITRY TATARNIKOV

In [1]:
import numpy as np

# CHAPTER 2 LEAST SQUARES ADJUSTMENT

## 2.1 Elementary Considerations

### 2.1.1 Statistical Nature of Surveying Measurements

- Different types of observations combined in one solution
- Continuous statistics is where measurements belong (imagine a machine that reads infinite decimals of a dist meas from tape.)
- Normal distribution

### 2.1.2 Observational Errors

- Random errors (ls takes care of it)
- Systematic errors
- Blunders (can be discovered with ls)

### 2.1.3 Accuracy and Precision

- Accuracy = how close to true value
- Precision = how close to mean observation

## 2.2 Stochastic and Mathematical Models

- stochastic = how reliable model is?
- mathematical = observables and unknowns
- var-covar matrix (multiply with 1/ A priori var) -> Q cofactor matrix (reverse) -> weight matrix P

## Self study

In [8]:
# Observed GNSS coordinates
b = np.array([
    [3654321.245, 255432.128, 5201234.512],
    [3654321.251, 255432.121, 5201234.519],
    [3654321.240, 255432.134, 5201234.508]
])
print(b)

[[3654321.245  255432.128 5201234.512]
 [3654321.251  255432.121 5201234.519]
 [3654321.24   255432.134 5201234.508]]


In [7]:
x0 = b[0].copy()
print("first X:", x0[0])
print("second X:", x0[1])
print("third X:", x0[2])

first X: 3654321.245
second X: 255432.128
third X: 5201234.512


- a=b+v, b observed gnss coord, v residuals, a corrected coords
- B is identity matrix, partial derivatives of observations
- A partial derivative of parameters
- w başlangıç model uyuşumsuzluğu

In [5]:
B = np.eye(9)
print(B)

[[1. 0. 0. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 1. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 1. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 0. 1.]]


In [20]:
A = np.array([
    [-1,  0,  0],
    [-1,  0,  0],
    [-1,  0,  0],
    [ 0, -1,  0],
    [ 0, -1,  0],
    [ 0, -1,  0],
    [ 0,  0, -1],
    [ 0,  0, -1],
    [ 0,  0, -1]
])

In [10]:
w_matrix = b - x0
print(w_matrix)

[[ 0.     0.     0.   ]
 [ 0.006 -0.007  0.007]
 [-0.005  0.006 -0.004]]


In [13]:
w = w_matrix.flatten()
print(w)

[ 0.     0.     0.     0.006 -0.007  0.007 -0.005  0.006 -0.004]


In [15]:
x, *_ = np.linalg.lstsq(A, -w, rcond=None)
print(x)

[ 0.     0.002 -0.001]


- Bv+Ax+w=0 and B=I so v=-Ax-w
- we want to minimize v2 so;
- min(vv^T) = (-Ax-w)^T(-Ax-w)
- (-times - = +)
- = (Ax+w)^T(Ax+w)
- = (x^T A^T + w^T) (Ax+w)
- = x^T A^T Ax + 2 x^T A^T w + w^T w
- (take derivative (x) and equalize to 0 for minimization)
- = 2 A^T Ax + 2 A^T w + 0 = 0
- A^T Ax must be equal to A^T w so;
- A^T Ax = - A^T w
- N=A^TA and u=-A^T w
- Nx = u 

In [21]:
A_T = A.T
print(A_T)

[[-1 -1 -1  0  0  0  0  0  0]
 [ 0  0  0 -1 -1 -1  0  0  0]
 [ 0  0  0  0  0  0 -1 -1 -1]]


In [22]:
N = A.T @ A
print(N)

[[3 0 0]
 [0 3 0]
 [0 0 3]]


In [23]:
u = -A.T @ w
print(u)

[ 0.     0.006 -0.003]


In [26]:
M = np.column_stack((N, u))
print(M)

[[ 3.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00]
 [ 0.00000000e+00  3.00000000e+00  0.00000000e+00  6.00000028e-03]
 [ 0.00000000e+00  0.00000000e+00  3.00000000e+00 -2.99999962e-03]]


In [27]:
x = np.zeros(3)
x[2] = M[2, 3] / M[2, 2]
print(x)

[ 0.     0.    -0.001]


In [28]:
x[1] = (M[1, 3] - M[1, 2] * x[2]) / M[1, 1]
print(x)

[ 0.     0.002 -0.001]


In [29]:
x[0] = (M[0, 3] - M[0, 1] * x[1] - M[0, 2] * x[2]) / M[0, 0]
print(x)

[ 0.     0.002 -0.001]


- x found with gauss elimination same as lsquares code in np

In [16]:
v = -A @ x - w
print(v)

[ 0.     0.     0.    -0.004  0.009 -0.005  0.004 -0.007  0.003]


In [18]:
xyz_adjusted = x0 + x
print(xyz_adjusted)

[3654321.245  255432.13  5201234.511]


In [19]:
print("Corrections in milimeter:",x * 1000)

Corrections in milimeter: [ 0.          2.00000009 -0.99999987]


- v residuals
- linearization (iterative)
- Jacobian = Coefficient matrix
- Bv + Ax + w = 0

## 2.3 Mixed Model

### 2.3.1 Linearization

### 2.3.2 Minimization and Solution

### 2.3.3 Cofactor Matrices

### 2.3.4 A Posteriori Variance of Unit Weight

### 2.3.5 Iterations

## 2.4 Sequential Mixed Model

## 2.5 Model Specifications